라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

데이터 로드

In [ ]:
df = pd.read_csv('../data/processed/df_merged.csv', encoding='cp949')
df

In [ ]:
print(sum(df.isna().sum()))
df.dropna(inplace=True)
print(sum(df.isna().sum()))

In [ ]:
df.columns

In [ ]:
# 주문별로 데이터 프레임 재구성
order_df = df.groupby(['idUser', 'idOrder']).agg({
    'Price': 'sum',
    'ItemCode': 'count',
    'OrderDT': 'first',
    'OrderYear': 'first',
    'OrderMonth': 'first',
    'OrderWeekday': 'first',
    'OrderHour': 'first',
    'IsLate': 'max'
}).rename(columns={
    'Price': 'order_amount',
    'ItemCode': 'item_cnt'
}).reset_index()
order_df

In [ ]:
df['OrderDT'] = pd.to_datetime(df['OrderDT'])
df['DeliveryDT'] = pd.to_datetime(df['DeliveryDT'])

월별/연도별 신규 고객수

In [ ]:
# 월별 신규 고객
first_order = order_df.groupby('idUser')['OrderDT'].min().reset_index()
first_order['OrderDT'] = pd.to_datetime(first_order['OrderDT'])
first_order['first_order_month'] = first_order['OrderDT'].dt.to_period('M')
print(first_order)

monthly_new_users = first_order.groupby('first_order_month')['idUser'].nunique()

monthly_new_users.plot(figsize=(10,5))
plt.title('월별 신규 고객 수')
plt.xlabel('월')
plt.ylabel('신규 고객 수')
plt.grid(True)
plt.show()

monthly_new_users

In [ ]:
df['OrderMonth'] = df['OrderDT'].dt.to_period('M')

item_first_month = df.groupby('ItemCode')['OrderMonth'].min().reset_index()
item_first_month.columns = ['ItemCode', 'first_month']

# 24년 1월에 처음 등장한 상품
new_items_jan = item_first_month[item_first_month['first_month'] == '2024-01']

print(len(new_items_jan))

new_items_detail = df[df['ItemCode'].isin(new_items_jan['ItemCode'])]

new_items_detail.groupby('ItemMiddleName').size().sort_values(ascending=False)

In [ ]:
# 23-12에서 24-01 사이 급증 상품
monthly_item = df.groupby(['OrderMonth','ItemName']).size().reset_index(name='cnt')

pivot_item = monthly_item.pivot(index='ItemName', columns='OrderMonth', values='cnt').fillna(0)
pivot_item.columns = pivot_item.columns.astype(str)

# 23-12 → 24-01만 사용
pivot_item['diff'] = pivot_item['2024-01'] - pivot_item['2023-12']
pivot_item['growth_rate'] = (pivot_item['2024-01'] + 1) / (pivot_item['2023-12'] + 1)

top_items = pivot_item[
    (pivot_item['2024-01'] > 50) & (pivot_item['diff'] > 30)
].sort_values('diff', ascending=False)
top_items[['2023-12', '2024-01']]

In [ ]:
first_order = df.sort_values('OrderDT').groupby('idUser').first().reset_index()

first_order['OrderMonth'] = first_order['OrderDT'].dt.to_period('M')

jan_first = first_order[first_order['OrderMonth'] == '2024-01']

jan_first['ItemMiddleName'].value_counts().head(10)

In [ ]:
# 23-12에서 24-01 사이 가격 변동 큰 제품
# 컬럼을 문자열로 변환 (Period → str)
pivot_item.columns = pivot_item.columns.astype(str)

# 2023-12 → 2024-01 diff 계산
pivot_item['diff'] = pivot_item['2024-01'] - pivot_item['2023-12']

# 성장률 계산 (0 나누기 방지용 +1 처리)
pivot_item['growth_rate'] = (pivot_item['2024-01'] + 1) / (pivot_item['2023-12'] + 1)

# 조건: 2024-01 가격이 50 이상 & diff가 30 이상
top_items = pivot_item[
    (pivot_item['2024-01'] > 50) & (pivot_item['diff'] > 30)
].sort_values('diff', ascending=False)

# 확인
top_items[['2023-12', '2024-01', 'diff', 'growth_rate']]

In [ ]:
# top_items[['2023-12','2024-01']].plot(kind='bar', figsize=(12,6))
# plt.title('23-12 → 24-01 판매량 비교 (급증 상품)')
# plt.xlabel('Item')
# plt.ylabel('판매량')
# plt.show()

In [ ]:
# 사용자별 첫 주문 월 계산
first_order_month = first_order.groupby('idUser')['OrderMonth'].min().reset_index()
first_order_month.columns = ['idUser', 'first_order_month']

# 원본 first_order와 합치기
first_order = first_order.merge(first_order_month, on='idUser', how='left')

# 2024-01 신규 고객 선택
jan_users = first_order[first_order['first_order_month'] == '2024-01']

# 2024-01 이전 고객
prev_users = first_order[first_order['first_order_month'] < '2024-01']

# 통계 확인
print("=== 1월 신규 고객 ===")
print(jan_users[['Age', 'FamilyCount']].describe())

print("\n=== 이전 고객 ===")
print(prev_users[['Age', 'FamilyCount']].describe())

print(jan_users['AgeGroup'].value_counts())

In [ ]:
# 연도별 신규 고객 수
first_order['year'] = first_order['OrderDT'].dt.year

yearly_new_users = first_order.groupby('year')['idUser'].nunique()
print(yearly_new_users)
yearly_new_users.plot(kind='bar', figsize=(8,5))
plt.title('연도별 신규 고객')
plt.xlabel('연도')
plt.ylabel('신규 고객 수')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# 연령대별 신규 유입
def age_segment_numeric(age):
    if age in [20, 30]:  # 20대 + 30대
        return '2030'
    elif age in [40, 50, 60]:  # 40~60대
        return '4060'
    else:
        return 'Other'

# AgeGroup이 숫자형이라면
first_order['AgeSegment'] = first_order['AgeGroup'].apply(age_segment_numeric)

# 연도 + 연령대별 신규 고객 수 (first_order_month 기준)
yearly_segment_users = first_order.groupby(['year','AgeSegment'])['idUser'].nunique().unstack(fill_value=0)

# 시각화
yearly_segment_users[['2030','4060']].plot(kind='bar', figsize=(10,6))
plt.title('연도별 신규 고객 수 (2030 vs 4060)')
plt.xlabel('연도')
plt.ylabel('신규 고객 수')
plt.xticks(rotation=0)
plt.show()

yearly_segment_users

In [ ]:
order_df['OrderDT'] = pd.to_datetime(order_df['OrderDT'])
order_df['order_month'] = order_df['OrderDT'].dt.to_period('M')

monthly_orders = order_df.groupby('order_month').size()
print(monthly_orders)

monthly_orders.index = monthly_orders.index.to_timestamp()

monthly_orders.plot(figsize=(10,5))
plt.title('월별 주문 수')
plt.xlabel('월')
plt.ylabel('주문 수')
plt.grid(True)
plt.show()

In [ ]:
order_df['order_year'] = order_df['OrderDT'].dt.year

yearly_orders = order_df.groupby('order_year').size()
yearly_orders

yearly_orders.plot(kind='bar', figsize=(8,5))
plt.title('연간 주문 수')
plt.xlabel('연도')
plt.ylabel('연도별 주문 수')
plt.xticks(rotation=0)
plt.show()

In [ ]:
new_users = first_order.groupby('first_order_month').size()

In [ ]:
df['OrderDT'] = pd.to_datetime(df['OrderDT'])
df['DeliveryDT'] = pd.to_datetime(df['DeliveryDT'])

# 주문 단위
order_df = df.groupby(['idUser', 'idOrder']).agg({
    'Price': 'sum',
    'ItemCode': 'count',
    'OrderDT': 'first',
    'IsLate': 'max'
}).rename(columns={
    'Price': 'order_amount',
    'ItemCode': 'item_cnt'
}).reset_index()

# 고객 단위
customer_df = order_df.groupby('idUser').agg({
    'idOrder': 'nunique',
    'order_amount': ['sum', 'mean'],
    'item_cnt': 'mean',
    'OrderDT': ['min', 'max'],
    'IsLate': 'sum'
})

customer_df.columns = [
    'order_cnt', 'total_amount', 'AOV',
    'avg_item_cnt', 'first_order', 'last_order', 'late_cnt'
]

customer_df = customer_df.reset_index()

# 고객 속성 붙이기
customer_info = df[['idUser','Gender','Age','AgeGroup','FamilyCount','MemberYN']].drop_duplicates()
customer_df = customer_df.merge(customer_info, on='idUser', how='left')

In [ ]:
member_summary = customer_df.groupby('MemberYN').agg({
    'order_cnt': 'mean',
    'total_amount': 'mean',
    'AOV': 'mean'
})

print(member_summary)

sns.boxplot(data=customer_df, x='MemberYN', y='total_amount')
plt.title('Total Spend by Membership')
plt.show()

In [ ]:
age_summary = customer_df.groupby('AgeGroup').agg({
    'order_cnt': 'mean',
    'total_amount': 'mean',
    'AOV': 'mean'
}).sort_index()

print(age_summary)

sns.barplot(data=customer_df, x='AgeGroup', y='total_amount')
plt.title('Spend by Age Group')
plt.show()

In [ ]:
summary_table = customer_df.describe()
print(summary_table)

연도별 객단가 변화

In [ ]:
# 연도별 총 매출과 주문 수 계산
yearly_stats = df.groupby('OrderYear').agg(
    total_revenue=('Price', 'sum'),
    total_orders=('idOrder', 'nunique')  # 주문 수 기준
).reset_index()

# 객단가 계산
yearly_stats['AOV'] = yearly_stats['total_revenue'] / yearly_stats['total_orders']
print(yearly_stats)

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(yearly_stats['OrderYear'], yearly_stats['AOV'], marker='o', linestyle='-')
plt.title('연도별 객단가 (AOV)')
plt.xlabel('연도')
plt.ylabel('객단가')
plt.grid(True)
plt.show()

In [ ]:
# 연도별 거래액
import matplotlib.pyplot as plt

# 1연도별 거래액 계산
yearly_revenue = df.groupby('OrderYear')['Price'].sum().reset_index()
yearly_revenue.rename(columns={'Price':'Revenue'}, inplace=True)

# 확인
print(yearly_revenue)

# 시각화
plt.figure(figsize=(8,5))
plt.bar(yearly_revenue['OrderYear'], yearly_revenue['Revenue'], color='skyblue')
plt.title('연간 거래액')
plt.xlabel('연도')
plt.ylabel('거래액')
plt.xticks(rotation=0)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# OrderDT를 datetime으로 변환
df['OrderDT'] = pd.to_datetime(df['OrderDT'])

# 연도와 날짜 컬럼 따로 만들기
df['OrderDate'] = df['OrderDT'].dt.date
df['OrderYear'] = df['OrderDT'].dt.year

# 날짜별 고유 주문 수 계산
daily_orders = df.groupby(['OrderYear', 'OrderDate'])['idOrder'].nunique().reset_index()
daily_orders.rename(columns={'idOrder':'DailyOrders'}, inplace=True)

# 연도별 일 평균 주문 건수 계산
avg_daily_orders_per_year = daily_orders.groupby('OrderYear')['DailyOrders'].mean().reset_index()
avg_daily_orders_per_year.rename(columns={'DailyOrders':'AvgDailyOrders'}, inplace=True)

print(avg_daily_orders_per_year)

# 시각화
plt.figure(figsize=(8,5))
plt.bar(avg_daily_orders_per_year['OrderYear'], avg_daily_orders_per_year['AvgDailyOrders'], color='lightgreen')
plt.title('연도별 일 평균 주문 건수')
plt.xlabel('연도')
plt.ylabel('일 평균 주문 수')
plt.xticks(rotation=0)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 연령대별, 제품별 구매 수 계산
age_item_count = df.groupby(['AgeGroup', 'ItemName'])['idOrder'].count().reset_index()
age_item_count.rename(columns={'idOrder':'PurchaseCount'}, inplace=True)

# 연령대별 Top 5 제품 선택
top_items_per_age = age_item_count.groupby('AgeGroup').apply(lambda x: x.nlargest(5, 'PurchaseCount')).reset_index(drop=True)

print(top_items_per_age)

# 시각화: 연령대별 Top 제품 막대 그래프
plt.figure(figsize=(12,8))
sns.barplot(
    data=top_items_per_age,
    x='PurchaseCount',
    y='ItemName',
    hue='AgeGroup'
)
plt.title('연령대별 Top 5 제품 구매 수')
plt.xlabel('구매 수')
plt.ylabel('제품명')
plt.legend(title='연령대')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 연령대별, 제품별 구매 수 계산
age_item_count = df.groupby(['AgeGroup', 'ItemLargeName'])['idOrder'].count().reset_index()
age_item_count.rename(columns={'idOrder':'PurchaseCount'}, inplace=True)

# 연령대별 Top 5 제품 선택
top_items_per_age = age_item_count.groupby('AgeGroup').apply(lambda x: x.nlargest(5, 'PurchaseCount')).reset_index(drop=True)

print(top_items_per_age)

# 시각화: 연령대별 Top 제품 막대 그래프
plt.figure(figsize=(12,8))
sns.barplot(
    data=top_items_per_age,
    x='PurchaseCount',
    y='ItemLargeName',
    hue='AgeGroup'
)
plt.title('연령대별 Top 5 제품 구매 수')
plt.xlabel('구매 수')
plt.ylabel('제품 대분류명')
plt.legend(title='연령대')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 연령대별, 제품별 구매 수 계산
age_item_count = df.groupby(['AgeGroup', 'ItemMiddleName'])['idOrder'].count().reset_index()
age_item_count.rename(columns={'idOrder':'PurchaseCount'}, inplace=True)

# 연령대별 Top 5 제품 선택
top_items_per_age = age_item_count.groupby('AgeGroup').apply(lambda x: x.nlargest(5, 'PurchaseCount')).reset_index(drop=True)

print(top_items_per_age)

# 시각화: 연령대별 Top 제품 막대 그래프
plt.figure(figsize=(12,8))
sns.barplot(
    data=top_items_per_age,
    x='PurchaseCount',
    y='ItemMiddleName',
    hue='AgeGroup'
)
plt.title('연령대별 Top 5 제품 구매 수')
plt.xlabel('구매 수')
plt.ylabel('제품 대분류명')
plt.legend(title='연령대')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 4060 세대 필터링
age_4060 = df[df['AgeGroup'].isin([40, 60])]

# 제품별 구매 건수 계산
item_count_4060 = age_4060.groupby('ItemLargeName')['idOrder'].count().reset_index()
item_count_4060.rename(columns={'idOrder':'PurchaseCount'}, inplace=True)

# Top 10 제품
top_items_4060 = item_count_4060.sort_values(by='PurchaseCount', ascending=False).head(10)
print(top_items_4060)

# 시각화
plt.figure(figsize=(10,6))
plt.barh(top_items_4060['ItemLargeName'], top_items_4060['PurchaseCount'], color='skyblue')
plt.xlabel('구매 건수')
plt.ylabel('제품명')
plt.title('4060 세대 Top 10 구매 제품')
plt.gca().invert_yaxis()  # 가장 많이 산 제품이 위로
plt.show()